# Load MultiVI-Splice model and plug in EasySci mudata

Uses the [multivi_tools_splicing](https://github.com/smritivaidyanathan/multivi_tools_splicing) workflow (MultiVI-Splice model from the forked **scvi-tools-splicing**). We load **one model at a time** and run inference on our EasySci mudata (rna + splicing modalities built in `01_make_mudata.ipynb`).

In [1]:
# Paths: EasySci mudata (from 01_make_mudata.ipynb) and one model at a time
MUDATA_PATH = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/EasySci2024/LeafletFA/mudata_easysci.h5mu"

# Training mudata: same structure/var order the model was trained on (required for load; then we can run inference on EasySci).
MUDATA_TRAINING_PATH = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/MODEL_INPUT/102025/train_70_30_model_ready_combined_gene_expression_aligned_splicing_20251009_024406_UPDATEDOBS.h5mu"

# Model: use a model trained with multivi_tools_splicing / scvi-tools-splicing.
MODEL_DIR = "/gpfs/commons/home/svaidyanathan/repos/SpliceVI/models/splicevi_basic_20251213_111454"  # folder containing model.pt

# Other model options (uncomment to use):
# MODEL_DIR = "/gpfs/commons/home/svaidyanathan/repos/SpliceVI/models/splicevi_basic_20260201_021528"
# MODEL_DIR = "/gpfs/commons/home/svaidyanathan/repos/SpliceVI/models/splicevi_basic_20260121_123415"
# MODEL_DIR = "/gpfs/commons/home/svaidyanathan/repos/SpliceVI/models/splicevi_basic_20260127_143801"

In [2]:
import mudata as md
import scvi

In [3]:
# Load EasySci mudata (rna + splicing, aligned to ss2 reference)
mdata = md.read_h5mu(MUDATA_PATH)
print(f"Loaded mudata: {mdata.n_obs} cells")
print(f"  rna: {mdata['rna'].shape}")
print(f"  splicing: {mdata['splicing'].shape}")
mdata

/gpfs/commons/home/kisaev/miniconda3/envs/scvi-env/lib/python3.12/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)


Loaded mudata: 4931 cells
  rna: (4931, 10572)
  splicing: (4931, 89831)


/gpfs/commons/home/kisaev/miniconda3/envs/scvi-env/lib/python3.12/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 4931 × 100403
  var:	'index', 'gene_name', 'gene_id', 'aging_lifespan_effect', 'aging_longevity_influence', 'aging_gene', 'ID', 'modality'
  2 modalities
    rna:	4931 x 10572
      obs:	'Type', 'kmeans_cluster', 'Main_cluster_name', 'Sex', 'Main_cluster_name_wkmeans', 'cell_id_index'
      var:	'index', 'gene_name', 'gene_id', 'mean_transcript_length', 'mean_intron_length', 'num_transcripts', 'gene_biotype', 'is_rRNA', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'RBP_gene', 'is_hvg', 'is_rbp', 'gene_name_upper', 'aging_gene_name', 'aging_lifespan_effect', 'aging_longevity_influence', 'aging_gene', 'ID', 'modality'
      layers:	'raw'
    splicing:	4931 x 89831
      obs:	'Main_cluster_name_wkmeans', 'Type', 'Main_cluster_name', 'Sex', 'kmeans_cluster', 'n_cells', 'cell_id_index'
      var:	'index', 'junction_id', 'event_id', 'splice_motif', 'annotation_status', 'gene_name', 'gene_id', 'num_junctions', 'position_off_5_prime', 'position_off_3_prime', 'CountJuncs', 'junction_id_index', 'n_cells_detected', 'confidence', 'aging_gene', 'aging_lifespan_effect', 'aging_longevity_influence', 'ID', 'modality'
      layers:	'cell_by_cluster_matrix', 'cell_by_junction_matrix'

In [4]:
# Load the mudata used for training (same var order/structure as the saved model; required for load).
training_mdata = md.read_h5mu(MUDATA_TRAINING_PATH)
print(f"Training mudata: {training_mdata.n_obs} cells, rna {training_mdata['rna'].n_vars}, splicing {training_mdata['splicing'].n_vars}")

/gpfs/commons/home/kisaev/miniconda3/envs/scvi-env/lib/python3.12/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/commons/home/kisaev/miniconda3/envs/scvi-env/lib/python3.12/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


Training mudata: 99620 cells, rna 10572, splicing 89831


In [5]:
# Discover the model class name saved in model.pt (fork uses a different class than MULTIVI).
import torch
_checkpoint = torch.load(f"{MODEL_DIR}/model.pt", map_location="cpu", weights_only=False)
_registry = _checkpoint.get("registry_", {})
# scvi-tools stores class name under "model_name" (or _model_name); print registry to debug if needed.
_model_name = _registry.get("model_name") or _registry.get("_model_name") or "?"
print("Saved model class name:", _model_name)
if _model_name == "?":
    print("Registry keys:", list(_registry.keys()))
ModelClass = getattr(scvi.model, _model_name)
print("Using class:", ModelClass)

Saved model class name: ?
Registry keys: []


AttributeError: module 'scvi.model' has no attribute '?'

In [ ]:
# Load the model with the correct class (from cell above) and training mudata so var names match.
model = ModelClass.load(MODEL_DIR, adata=training_mdata)
print(f"Loaded model from {MODEL_DIR}")

## Plug mudata into the model

The model was trained on mudata with modalities registered via `setup_mudata()`. Our EasySci mudata has the same structure (rna + splicing, same var order as training). We either:

- **Same structure as training:** run `setup_mudata()` with the same modality keys and layers so the model can run inference, or  
- **Query / transfer:** use the model’s `prepare_query_mudata()` if it supports it (reference → query integration).

Checking whether the model has a `prepare_query_mudata` method (transfer learning) or we use `setup_mudata` and then get latent.

In [ ]:
# Option A: If our mudata matches training (same genes/junctions), register it and get latent.
# Replace layer names if your training used different layers (e.g. "raw" for rna, "cell_by_junction_matrix" for splicing).
# Uncomment and adjust after confirming the setup used in multivi_tools_splicing:

# scvi.model.MULTIVI.setup_mudata(
#     mdata,
#     rna_layer="raw",           # or None for mdata["rna"].X
#     protein_layer=None,
#     batch_key=None,            # optional, e.g. "batch" or "donor_id"
# )
# model.get_latent_representation(mdata)

# Option B: If the loaded model supports query integration (e.g. transfer from reference):
# mdata = model.prepare_query_mudata(mdata)
# model.get_latent_representation(mdata)

# For now, just verify model and mudata are loaded and show model's expected setup (if stored in the save dir).
import os
print("Contents of model dir:", os.listdir(MODEL_DIR))

In [ ]:
# Next: align setup with multivi_tools_splicing (multi_vi_splice_notebook.ipynb).
# Once you know the exact setup_mudata() args (modality names, layers), uncomment Option A in the cell above
# and run get_latent_representation(); then store latents in mdata.obsm["X_multivi"] for downstream use.